# Profiling Python Code
## Motivation

When optimizing scientific or high-performance Python code, intuition is usually wrong.

Many developers optimize:
- the wrong function,
- the wrong loop,
- or code that contributes almost nothing to total runtime.

Profiling answers a fundamental question:

> "Where is the program actually spending time?"

Optimization without profiling is often wasted effort.

The standard workflow is:

1. Measure runtime naively
2. Benchmark more rigorously
3. Profile function-level hotspots
4. Profile line-by-line bottlenecks
5. Analyze memory behavior
6. Optimize only the dominant costs

The standard Python libraries that we can use to profile are:

In [ ]:
import time
import timeit
import cProfile
import pstats

## Example Problem

We begin with a deliberately inefficient implementation.

In [25]:
def slow_sum_of_squares(n):
    total = 0

    for i in range(n):
        total += i ** 2

    return total

## 1. Naive Timing with `time`

The simplest timing strategy uses wall-clock measurements.

This gives a first estimate of runtime, but single measurements are noisy.

In [26]:
for i in range(5):   
    start = time.perf_counter()
    slow_sum_of_squares(10_000_000)
    end = time.perf_counter()
    print(f"Elapsed time run {i}: {end - start:.6f} seconds")

Elapsed time run 0: 0.569279 seconds
Elapsed time run 1: 0.521319 seconds
Elapsed time run 2: 0.539595 seconds
Elapsed time run 3: 0.532943 seconds
Elapsed time run 4: 0.524078 seconds


### Why `perf_counter()`?

Python provides multiple clocks.

`time.perf_counter()` is preferred because:
- high precision,
- includes sleep time,
- designed for benchmarking.

Avoid:
- `time.time()`
- manual stopwatch approaches

### Problem with Naive Timing

Single-run measurements are noisy.

Runtime may vary due to:
- CPU frequency scaling,
- cache effects,
- OS scheduling,
- background processes,
- JIT warmups.

We need repeated measurements.

## 2. Better Benchmarking with `timeit`

`timeit` executes code multiple times and reports more reliable timing statistics.

This is the preferred approach for:
- kernels,
- numerical loops,
- micro-optimizations.

In [27]:
execution_time = timeit.timeit(
    "slow_sum_of_squares(10_000_000)",
    globals=globals(),
    number=5
)

print(f"Average execution time: {execution_time / 5:.6f} seconds")

Average execution time: 0.598875 seconds


In [28]:
%timeit slow_sum_of_squares(10_000_000)

569 ms ± 41.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


## Timing Multiple Variants

Suppose we compare:
- pure Python,
- NumPy vectorization.

In [29]:
import numpy as np
def python_sum_of_squares(n):
    total = 0

    for i in range(n):
        total += i ** 2

    return total


def numpy_sum_of_squares(n):
    x = np.arange(n)
    return np.sum(x ** 2)

In [30]:
python_time = timeit.timeit(
    "python_sum_of_squares(10_000_000)",
    globals=globals(),
    number=3
)

numpy_time = timeit.timeit(
    "numpy_sum_of_squares(10_000_000)",
    globals=globals(),
    number=3
)

print(f"Python average : {python_time / 3:.6f} s")
print(f"NumPy average  : {numpy_time / 3:.6f} s")

Python average : 0.575032 s
NumPy average  : 0.076677 s


In [31]:
%timeit python_sum_of_squares(10_000_000)
%timeit numpy_sum_of_squares(10_000_000)

557 ms ± 59.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
46.2 ms ± 3.59 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


NumPy is typically much faster because:
- loops execute in optimized C,
- vectorized operations reduce Python interpreter overhead,
- memory access patterns are optimized.

This demonstrates a central HPC principle:

> Python itself is often not slow.
> Python loops are slow.

## 3. Function-Level Profiling with `cProfile`

Timing tells us *how long* something takes.

Profiling tells us:
- which functions dominate runtime,
- how often they are called,
- cumulative execution costs.

In [32]:
def expensive_operation():
    total = 0

    for i in range(1_000_000):
        total += np.sqrt(i)

    return total


def main():
    for _ in range(5):
        expensive_operation()

In [33]:
cProfile.run("main()")

         542 function calls (533 primitive calls) in 4.856 seconds

   Ordered by: standard name

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        5    4.210    0.842    4.210    0.842 1757648358.py:1(expensive_operation)
        1    0.000    0.000    4.775    4.775 1757648358.py:10(main)
        2    0.000    0.000    0.000    0.000 <frozen abc>:121(__subclasscheck__)
        4    0.000    0.000    0.000    0.000 <frozen importlib._bootstrap>:1390(_handle_fromlist)
        1    0.000    0.000    4.775    4.775 <string>:1(<module>)
        1    0.000    0.000    0.032    0.032 asyncio.py:206(_handle_events)
        1    0.001    0.001    0.001    0.001 asyncio.py:231(add_callback)
        4    0.000    0.000    0.000    0.000 attrsettr.py:43(__getattr__)
        4    0.000    0.000    0.000    0.000 attrsettr.py:66(_get_attr_opt)
        1    0.000    0.000    0.000    0.000 base_events.py:1895(_add_callback)
        1    0.000    0.000    0.000    0.000

### Understanding the Output

Typical columns:

| Column | Meaning |
|---|---|
| ncalls | Number of calls |
| tottime | Time spent inside function only |
| cumtime | Time including subcalls |
| filename:lineno(function) | Function identifier |

Key insight:
- `tottime` isolates local work,
- `cumtime` reveals full call cost.

### Saving Profile Results

Profiles can be saved for later inspection.

In [34]:
cProfile.run("main()", "profile_results.prof")

In [35]:
stats = pstats.Stats("profile_results.prof")

stats.sort_stats("cumtime").print_stats(10)

Wed May 27 17:28:47 2026    profile_results.prof

         489 function calls (484 primitive calls) in 4.740 seconds

   Ordered by: cumulative time
   List reduced from 104 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      2/1    0.039    0.020    4.643    4.643 {built-in method builtins.exec}
        1    0.000    0.000    4.643    4.643 <string>:1(<module>)
        1    0.283    0.283    4.643    4.643 /tmp/ipykernel_10671/1757648358.py:10(main)
        5    2.538    0.508    2.538    0.508 /tmp/ipykernel_10671/1757648358.py:1(expensive_operation)
        4    1.821    0.455    1.821    0.455 {built-in method time.sleep}
        2    0.004    0.002    0.040    0.020 /usr/lib/python3.12/asyncio/events.py:86(_run)
        2    0.000    0.000    0.035    0.018 {method 'run' of '_contextvars.Context' objects}
        2    0.000    0.000    0.035    0.018 /home/jkhansell/Documents/ULeadParallelComp2026/.venv/lib/python3.12/site-p

In [37]:
!uv pip install line_profiler
%reload_ext line_profiler

Using Python 3.12.3 environment at: /home/jkhansell/Documents/ULeadParallelComp2026/.venv
Checked 1 package in 3ms


In [38]:
def compute():
    total = 0

    for i in range(1_000_000):
        total += np.sqrt(i)

    return total

In [39]:
%lprun -f compute compute()

Timer unit: 1e-09 s

Total time: 1.41287 s
File: /tmp/ipykernel_10671/4102803027.py
Function: compute at line 1

Line #      Hits         Time  Per Hit   % Time  Line Contents
     1                                           def compute():
     2         1       5900.0   5900.0      0.0      total = 0
     3                                           
     4   1000001  266896033.0    266.9     18.9      for i in range(1_000_000):
     5   1000000 1145964553.0   1146.0     81.1          total += np.sqrt(i)
     6                                           
     7         1       3257.0   3257.0      0.0      return total

## 5. Memory Profiling

Performance is not only about compute.

Memory issues include:
- unnecessary allocations,
- temporary arrays,
- copies,
- fragmentation,
- cache inefficiency.

In [41]:
!uv pip install memory_profiler
%reload_ext memory_profiler

Using Python 3.12.3 environment at: /home/jkhansell/Documents/ULeadParallelComp2026/.venv
Checked 1 package in 4ms


In [42]:
from memory_profiler import profile

In [43]:
%%writefile memory_example.py

import numpy as np

def allocate_memory():
    x = np.random.rand(10_000_000)
    y = np.random.rand(10_000_000)

    return x + y

Overwriting memory_example.py


In [44]:
from memory_example import allocate_memory

In [45]:
%mprun -f allocate_memory allocate_memory()

Filename: /home/jkhansell/Documents/ULeadParallelComp2026/src/performance_analysis/memory_example.py

Line #    Mem usage    Increment  Occurrences   Line Contents
     4    163.7 MiB    163.7 MiB           1   def allocate_memory():
     5    240.2 MiB     76.4 MiB           1       x = np.random.rand(10_000_000)
     6    316.4 MiB     76.2 MiB           1       y = np.random.rand(10_000_000)
     7                                         
     8    392.7 MiB     76.3 MiB           1       return x + y

## Why Bottlenecks Matter

Suppose a simulation code spends:
- 95% of time in one loop,
- 5% elsewhere.

Optimizing everything equally is irrational.

Amdahl’s Law implies that optimization effort should focus on the dominant cost first.

$$
S = \frac{1}{s+(1-s)/N}
$$

Where: 

- $s$: Serial fraction $1-p$
- $N$: Processor count

## Practical Optimization Workflow

A good workflow is:

1. Write correct code
2. Measure runtime
3. Profile bottlenecks
4. Optimize dominant kernels
5. Re-profile
6. Repeat

Never optimize blindly.

## Typical Scientific Python Bottlenecks

| Problem | Typical Solution |
|---|---|
| Python loops | NumPy vectorization |
| Repeated allocations | Buffer reuse |
| Small kernels | Numba |
| Slow Python dispatch | Cython/Pythran |
| Memory bandwidth limits | Blocking/cache locality |
| Serial execution | Parallelization |

## Advanced Profilers

### Snakeviz

Visualize `cProfile` results interactively.

Install:

```bash
pip install snakeviz

snakeviz profile_results.prof
```
### Scalene

```bash
pip install scalene

scalene my_script.py
```


## Final Takeaways

Key lessons:

- Measure before optimizing
- Timing is not profiling
- Repeated measurements matter
- Bottlenecks are usually localized
- Memory performance matters
- Profiling guides optimization strategy

The fastest optimization is:
> removing unnecessary work entirely.